# The Visual Storyteller: Image Captioning
## Notebook 1: Data Loading & Training

This notebook handles:
1. **Data Loading** - Read images and captions from disk
2. **Preprocessing** - Image resizing, tokenization, vocabulary building
3. **Model Definition** - EfficientNet-B0 encoder + LSTM + Bahdanau-style Attention decoder
4. **Training** - Full training loop with validation and early stopping
5. **Model Saving** - Save trained artifacts for the inference notebook

**Environment:** local Jupyter Notebook (no Google Colab / Drive dependency). All paths are
relative to this notebook's location inside `notebooks/`, and point at the `data/` and
`models/` folders described in the project README.


## 1. Setup & Dependencies

In [1]:
#   pip install -r ../requirements.txt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torchvision.models as models

import numpy as np
import pandas as pd
from PIL import Image
import os
import pickle
from collections import Counter
from tqdm import tqdm
import json
import random
import warnings
warnings.filterwarnings('ignore')

def set_seed(seed=42):
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    np.random.seed(seed)
    random.seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


Device: cpu


## 2. Load Data (Local Paths)

In [2]:
DATA_DIR = '../data'
MODELS_DIR = '../models'

images_dir = os.path.join(DATA_DIR, 'Images')
captions_file = os.path.join(DATA_DIR, 'captions.txt')

os.makedirs(MODELS_DIR, exist_ok=True)

print(f"Data directory: {DATA_DIR}")
print(f"Images directory exists: {os.path.exists(images_dir)}")
print(f"Captions file exists: {os.path.exists(captions_file)}")
if os.path.exists(images_dir):
    print(f"Number of images: {len(os.listdir(images_dir))}")
else:
    raise FileNotFoundError(
        f"Could not find {images_dir}. Place your dataset images in data/Images/ "
        f"and the captions file at data/captions.txt before running this notebook."
    )


Data directory: ../data
Images directory exists: True
Captions file exists: True
Number of images: 8091


## 3. Load and Parse Captions

In [3]:
def load_captions(captions_file):
    captions_dict = {}

    with open(captions_file, 'r', encoding='utf-8') as f:
        # Skip header
        next(f)

        for line in f:
            parts = line.strip().split(',')
            if len(parts) >= 2:
                image_name = parts[0]
                caption = ','.join(parts[1:]).strip()  # handle commas in captions

                if image_name not in captions_dict:
                    captions_dict[image_name] = []
                captions_dict[image_name].append(caption)

    return captions_dict

captions_dict = load_captions(captions_file)
print(f"Total unique images: {len(captions_dict)}")
print(f"Total captions: {sum(len(v) for v in captions_dict.values())}")

sample_img = list(captions_dict.keys())[0]
print(f"\nSample image: {sample_img}")
print(f"Captions for this image:")
for i, cap in enumerate(captions_dict[sample_img], 1):
    print(f"  {i}. {cap}")


Total unique images: 8091
Total captions: 40455

Sample image: 1000268201_693b08cb0e.jpg
Captions for this image:
  1. A child in a pink dress is climbing up a set of stairs in an entry way .
  2. A girl going into a wooden building .
  3. A little girl climbing into a wooden playhouse .
  4. A little girl climbing the stairs to her playhouse .
  5. A little girl in a pink dress going into a wooden cabin .


## 4. Build Vocabulary & Tokenizer

In [4]:
class Tokenizer:

    def __init__(self):
        self.word2idx = {}
        self.idx2word = {}
        self.word_freq = Counter()

        self.PAD_IDX = 0
        self.START_IDX = 1
        self.END_IDX = 2
        self.UNK_IDX = 3

        self.word2idx['<PAD>'] = self.PAD_IDX
        self.word2idx['<START>'] = self.START_IDX
        self.word2idx['<END>'] = self.END_IDX
        self.word2idx['<UNK>'] = self.UNK_IDX

        self.idx2word = {v: k for k, v in self.word2idx.items()}

    def build_vocab(self, captions_dict, min_freq=1):
        for captions in captions_dict.values():
            for caption in captions:
                words = self.tokenize_caption(caption)
                self.word_freq.update(words)

        next_idx = 4
        for word, freq in self.word_freq.most_common():
            if freq >= min_freq:
                self.word2idx[word] = next_idx
                self.idx2word[next_idx] = word
                next_idx += 1

        print(f"Vocabulary size: {len(self.word2idx)}")

    @staticmethod
    def tokenize_caption(caption):
        caption = caption.lower()
        caption = caption.replace('.', '').replace(',', '').replace('!', '').replace('?', '')
        words = caption.split()
        return words

    def encode(self, caption, max_length=30):
        
        words = self.tokenize_caption(caption)
        tokens = [self.START_IDX] 

        for word in words[:max_length-2]:
            tokens.append(self.word2idx.get(word, self.UNK_IDX))

        tokens.append(self.END_IDX)

        while len(tokens) < max_length:
            tokens.append(self.PAD_IDX)

        return tokens[:max_length]

    def decode(self, tokens):
        words = []
        for token in tokens:
            if token == self.START_IDX:
                continue
            elif token == self.END_IDX:
                break
            elif token == self.PAD_IDX:
                continue
            else:
                words.append(self.idx2word.get(token, '<UNK>'))

        return ' '.join(words)

# build tokenizer
tokenizer = Tokenizer()
tokenizer.build_vocab(captions_dict, min_freq=5)
print(f"\nTokenizer ready. Vocabulary size: {len(tokenizer.word2idx)}")

# test tokenizer
test_caption = captions_dict[sample_img][0]
encoded = tokenizer.encode(test_caption)
decoded = tokenizer.decode(encoded)
print(f"\nTest encoding:")
print(f"  Original: {test_caption}")
print(f"  Encoded: {encoded[:10]}... (length: {len(encoded)})")
print(f"  Decoded: {decoded}")


Vocabulary size: 3035

Tokenizer ready. Vocabulary size: 3035

Test encoding:
  Original: A child in a pink dress is climbing up a set of stairs in an entry way .
  Encoded: [1, 4, 45, 5, 4, 93, 174, 8, 122, 55]... (length: 30)
  Decoded: a child in a pink dress is climbing up a set of stairs in an <UNK> way


## 5. Create Dataset & DataLoader

In [5]:
class ImageCaptionDataset(Dataset):

    def __init__(self, image_names, captions_dict, images_dir, tokenizer,
                 max_caption_length=30, transform=None):
        self.image_names = image_names
        self.captions_dict = captions_dict
        self.images_dir = images_dir
        self.tokenizer = tokenizer
        self.max_caption_length = max_caption_length
        self.transform = transform

    def __len__(self):
        return len(self.image_names)

    def __getitem__(self, idx):
        image_name = self.image_names[idx]
        image_path = os.path.join(self.images_dir, image_name)

        try:
            image = Image.open(image_path).convert('RGB')
        except Exception as e:
            print(f"Error loading {image_path}: {e}")
            
            image = Image.new('RGB', (224, 224), color='black')

        if self.transform:
            image = self.transform(image)

        captions = self.captions_dict[image_name]
        caption = random.choice(captions)

        # encode caption
        encoded_caption = self.tokenizer.encode(caption, self.max_caption_length)

        return {
            'image': image,
            'caption': torch.tensor(encoded_caption, dtype=torch.long),
            'image_name': image_name
        }

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.CenterCrop((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Transforms defined.")


Transforms defined.


## 6. Train/Val/Test Split

In [6]:
# unique image names
all_images = list(captions_dict.keys())
random.shuffle(all_images)

# split: 80% train, 10% val, 10% test
total_images = len(all_images)
train_size = int(0.8 * total_images)
val_size = int(0.1 * total_images)

train_images = all_images[:train_size]
val_images = all_images[train_size:train_size + val_size]
test_images = all_images[train_size + val_size:]

print(f"Train images: {len(train_images)} ({len(train_images) * 5} captions)")
print(f"Val images: {len(val_images)} ({len(val_images) * 5} captions)")
print(f"Test images: {len(test_images)} ({len(test_images) * 5} captions)")

# Create datasets
train_dataset = ImageCaptionDataset(
    train_images, captions_dict, images_dir, tokenizer,
    max_caption_length=30, transform=train_transform
)

val_dataset = ImageCaptionDataset(
    val_images, captions_dict, images_dir, tokenizer,
    max_caption_length=30, transform=val_transform
)

test_dataset = ImageCaptionDataset(
    test_images, captions_dict, images_dir, tokenizer,
    max_caption_length=30, transform=val_transform
)

print("\nDatasets created successfully.")


Train images: 6472 (32360 captions)
Val images: 809 (4045 captions)
Test images: 810 (4050 captions)

Datasets created successfully.


## 7. Create DataLoaders

In [7]:
train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64,
    shuffle=False,
    num_workers=2,
    pin_memory=True
)

print(f"Train loader batches: {len(train_loader)}")
print(f"Val loader batches: {len(val_loader)}")
print(f"Test loader batches: {len(test_loader)}")

batch = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  Images: {batch['image'].shape}")
print(f"  Captions: {batch['caption'].shape}")
print(f"  Image names: {len(batch['image_name'])}")


Train loader batches: 102
Val loader batches: 13
Test loader batches: 13

Batch shapes:
  Images: torch.Size([64, 3, 224, 224])
  Captions: torch.Size([64, 30])
  Image names: 64


## 8. Model Architecture

Encoder: EfficientNet-B0 (ImageNet-pretrained, frozen) producing a 7x7x1280 spatial feature map.
Decoder: LSTM with Bahdanau-style additive attention over the 49 spatial regions, conditioned on the current hidden state.

In [8]:
class Attention(nn.Module):
    def __init__(self, hidden_size, feature_size):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size)
        self.U = nn.Linear(feature_size, hidden_size)
        self.V = nn.Linear(hidden_size, 1)

    def forward(self, hidden, encoder_output):
        hidden_proj = self.W(hidden).unsqueeze(1)
        feat_proj = self.U(encoder_output)
        score = self.V(torch.tanh(hidden_proj + feat_proj)).squeeze(2)
        attention_weight = torch.softmax(score, dim=1)
        context = torch.sum(attention_weight.unsqueeze(2) * encoder_output, dim=1)
        return context, attention_weight


class ImageCaptioningModel(nn.Module):
    def __init__(self, vocab_size, embedding_dim=300, hidden_size=600, feature_size=1280):
        super().__init__()
        self.vocab_size = vocab_size
        self.embedding_dim = embedding_dim
        self.hidden_size = hidden_size
        self.feature_size = feature_size

        try:
            weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1
            base = models.efficientnet_b0(weights=weights)
        except AttributeError:
            base = models.efficientnet_b0(pretrained=True)

        self.encoder = base.features
        for p in self.encoder.parameters():
            p.requires_grad = False

        self.feature_proj = nn.Linear(feature_size, feature_size)
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.dropout = nn.Dropout(0.25)

        self.lstm = nn.LSTM(embedding_dim + feature_size, hidden_size, batch_first=True)
        self.attention = Attention(hidden_size, feature_size)
        self.fc_out = nn.Linear(hidden_size + feature_size, vocab_size)

    def forward(self, images, captions):
        batch_size = images.size(0)
        max_length = captions.size(1)

        with torch.no_grad():
            feat_map = self.encoder(images)                     # (B, 1280, 7, 7)
        feat_map = feat_map.flatten(2).permute(0, 2, 1)          # (B, 49, 1280)
        encoder_features = self.feature_proj(feat_map)
        encoder_features = self.dropout(encoder_features)

        h = torch.zeros(1, batch_size, self.hidden_size, device=images.device)
        c = torch.zeros(1, batch_size, self.hidden_size, device=images.device)

        embedded = self.dropout(self.embedding(captions))

        outputs = []
        for t in range(max_length):
            hidden_query = h.squeeze(0)
            context, _ = self.attention(hidden_query, encoder_features)
            lstm_input = torch.cat([embedded[:, t:t+1, :], context.unsqueeze(1)], dim=2)
            lstm_out, (h, c) = self.lstm(lstm_input, (h, c))
            combined = torch.cat([lstm_out[:, 0, :], context], dim=1)
            combined = self.dropout(combined)
            outputs.append(self.fc_out(combined).unsqueeze(1))

        return torch.cat(outputs, dim=1)

    def get_encoder_features(self, image):
        with torch.no_grad():
            feat_map = self.encoder(image)
            feat_map = feat_map.flatten(2).permute(0, 2, 1)
            encoder_features = self.feature_proj(feat_map)
        return encoder_features

print("Model architecture: EfficientNet-B0 encoder + LSTM/Attention decoder (49 spatial regions).")


Model architecture: EfficientNet-B0 encoder + LSTM/Attention decoder (49 spatial regions).


In [9]:
print(f"Vocabulary size that the model will be built with: {len(tokenizer.word2idx)}")

Vocabulary size that the model will be built with: 3035


## 9. Training Setup

In [10]:
model = ImageCaptioningModel(
    vocab_size=len(tokenizer.word2idx),
    embedding_dim=300,
    hidden_size=600,
    feature_size=1280
)
model = model.to(device)

loss_fn = nn.CrossEntropyLoss(ignore_index=0, label_smoothing=0.1)

decoder_params = [
    {'params': model.embedding.parameters()},
    {'params': model.lstm.parameters()},
    {'params': model.attention.parameters()},
    {'params': model.fc_out.parameters()},
    {'params': model.feature_proj.parameters()}
]

optimizer = torch.optim.Adam(decoder_params, lr=0.001, weight_decay=5e-6)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=60, eta_min=5e-7)



## 10. Training Loop with Early Stopping

In [11]:
def train_epoch(model, train_loader, loss_fn, optimizer, device):
    model.train()
    total_loss = 0

    for batch in tqdm(train_loader, desc='Training'):
        images = batch['image'].to(device)
        captions = batch['caption'].to(device)

        outputs = model(images, captions)

        loss = loss_fn(
            outputs[:, :-1].reshape(-1, model.vocab_size),
            captions[:, 1:].reshape(-1)
        )

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(train_loader)


def validate(model, val_loader, loss_fn, device):
    model.eval()
    total_loss = 0

    with torch.no_grad():
        for batch in tqdm(val_loader, desc='Validating'):
            images = batch['image'].to(device)
            captions = batch['caption'].to(device)

            outputs = model(images, captions)

            loss = loss_fn(
                outputs[:, :-1].reshape(-1, model.vocab_size),
                captions[:, 1:].reshape(-1)
            )
            total_loss += loss.item()

    return total_loss / len(val_loader)


num_epochs = 60
patience = 12
best_val_loss = float('inf')
patience_counter = 0

training_history = {
    'train_loss': [],
    'val_loss': []
}

best_model_path = os.path.join(MODELS_DIR, 'best_model.pt')

print(f"Starting training for {num_epochs} epochs...\n")
for epoch in range(num_epochs):
    print(f"Epoch {epoch+1}/{num_epochs}")

    train_loss = train_epoch(model, train_loader, loss_fn, optimizer, device)
    training_history['train_loss'].append(train_loss)

    val_loss = validate(model, val_loader, loss_fn, device)
    training_history['val_loss'].append(val_loss)

    print(f"Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f}")

    scheduler.step()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0

        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': val_loss,
        }, best_model_path)
        print(f"  Best model saved to {best_model_path}")
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            break

    print()


Starting training for 60 epochs...

Epoch 1/60


Training:   0%|                                        | 0/102 [00:04<?, ?it/s]


KeyboardInterrupt: 

## 11. Save Training Artifacts

In [ ]:
# save tokenizer
tokenizer_path = os.path.join(MODELS_DIR, 'tokenizer.pkl')
with open(tokenizer_path, 'wb') as f:
    pickle.dump(tokenizer, f)
print(f"Tokenizer saved to {tokenizer_path}")

test_data_path = os.path.join(MODELS_DIR, 'test_data.json')
test_data = {
    'test_images': test_images,
    'captions_dict': {img: captions_dict[img] for img in test_images}
}
with open(test_data_path, 'w') as f:
    json.dump(test_data, f)
print(f"Test data saved to {test_data_path}")

history_path = os.path.join(MODELS_DIR, 'training_history.pkl')
with open(history_path, 'wb') as f:
    pickle.dump(training_history, f)
print(f"Training history saved to {history_path}")

print("\nAll training artifacts saved!")
print(f"  - Model:        {best_model_path}")
print(f"  - Tokenizer:    {tokenizer_path}")
print(f"  - Test data:    {test_data_path}")
print(f"  - History:      {history_path}")


## 12. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
plt.plot(training_history['train_loss'], label='Train Loss', marker='o')
plt.plot(training_history['val_loss'], label='Val Loss', marker='s')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training History')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(MODELS_DIR, 'training_history.png'), dpi=150)
plt.show()

print(f"\nFinal Results:")
print(f"  Best Val Loss: {best_val_loss:.4f}")
print(f"  Final Train Loss: {training_history['train_loss'][-1]:.4f}")
print(f"  Total epochs trained: {len(training_history['train_loss'])}")
